## 🎶 Jukebox AI 🎶

K.R.A.P. Records는 음악 및 시장 트렌드를 분석하여 발매 전략을 최적화할 수 있는 예측 머신러닝 모델을 개발하고자 합니다. 이를 위해 두 개의 데이터셋을 보유하고 있습니다. 하나는 국가별 일간 Top 50 곡의 모든 음악적 특성에 대한 정보를 담고 있으며, 다른 하나는 트랙의 이름, 국가, 아티스트, 발매일 정보를 포함하고 있습니다.

첫 단계로, 탐색적 데이터 분석(Exploratory Data Analysis, EDA)을 수행할 예정입니다. 머신러닝에서 EDA는 데이터셋의 주요 특성을 요약하고, 주로 시각적 방법을 활용해 데이터를 분석하는 과정입니다. 이를 통해 데이터의 구조를 이해하고, 패턴을 발견하며, 이상치를 식별하고, 머신러닝 모델을 적용하기 전에 가설을 수립할 수 있습니다.

이번 분석의 목표는 곡의 특성들 간의 상관관계를 파악하여 어떤 요소들이 곡의 인기에 영향을 미치는지를 이해하는 것입니다. 이러한 관계를 식별함으로써 서로 다른 변수들이 연결되어 있는지, 그리고 그 연결의 강도가 어느 정도인지를 판단할 수 있습니다.

![inner-loop-1](../images/inner-loop-1.png)


### 🐠 패키지 설치 및 불러오기

노트북을 개발해 나가는 과정에서 필요한 패키지들을 설치하고 불러와야 합니다. 기본으로 사용할 수 있도록 몇 개의 시작용 셀을 미리 만들어 두었지만, 노트북을 진행하면서 추가로 셀을 작성해야 합니다.

In [ ]:
!pip install seaborn
# 작업을 진행하면서 필요한 모듈을 이 셀에서 설치할 수 있습니다

In [ ]:
import seaborn as sns
import pandas as pd
import matplotlib.pyplot as plt
# 여기에서 필요한 추가 모듈과 클래스를 임포트하세요 — 수정 후에는 반드시 이 셀을 다시 실행하세요!

### 📦 데이터 불러오기

이제 실제 데이터로 작업을 시작해 보겠습니다.
우리는 두 개의 서로 다른 데이터셋을 가지고 있습니다. 첫 번째 데이터셋은 곡의 음악적 특성만을 포함하고 있으며, 두 번째 데이터셋은 해당 곡의 인기도 정보와 어떤 국가에서 인기가 있는지를 담고 있습니다.

머신러닝 모델을 구축하기 위해서는 이 두 가지 정보가 모두 필요하므로, 두 데이터셋을 병합한 후 전체 데이터를 대상으로 분석을 진행할 예정입니다.

데이터셋은 GitHub에 저장되어 있습니다. 만약 로컬 환경에 데이터가 없다면, GitHub에서 직접 데이터를 불러오는 방식을 사용하여 로드할 수 있습니다.

In [ ]:
# 데이터셋 불러오기
song_properties_data = pd.read_parquet('https://github.com/rhoai-mlops/jukebox/raw/refs/heads/main/99-data_prep/song_properties.parquet')
song_properties_data

In [ ]:
song_rankings_data = pd.read_parquet('https://github.com/rhoai-mlops/jukebox/raw/refs/heads/main/99-data_prep/song_rankings.parquet')
song_rankings_data

In [ ]:
# 데이터셋 합치기
data = pd.merge(song_properties_data.drop(["snapshot_date", "name", "artists"], axis=1), song_rankings_data, on='spotify_id')
data.head()

In [ ]:
# 데이터프레임을 전치(transpose)하여
# 각 컬럼의 통계 정보를 더 쉽게 비교할 수 있도록 합니다.
# 가로로 긴 테이블을 좌우로 스크롤하는 대신, 통계값을 세로 방향으로 읽을 수 있습니다.
data.describe().T

In [ ]:
# 결측값(missing values)이 포함된 행을 데이터프레임에서 제거합니다.
# 결측값을 제거함으로써 학습 및 분석에 사용되는 데이터셋의
# 완전성과 일관성을 유지할 수 있습니다.
data = data.dropna()
data.head()

In [ ]:
# 데이터가 수집된 국가 코드 목록
data['country'].unique()

In [ ]:
# 국가 코드는 문자열입니다. 여기서 각 국가에 고유한 번호를 부여하고 있습니다
# 따라서 문자열 대신 국가를 숫자로 처리할 수 있습니다
# 컴퓨터는 문자열을 좋아하지 않기 때문입니다
mapping = {c:i for i, c in enumerate(data['country'].unique())}
mapping

In [ ]:
pd.set_option('future.no_silent_downcasting', True)
data["country"] = data['country'].replace(
   mapping
).astype(int)

이제 데이터가 더 나은 형태가 되었으므로 상관관계를 찾아봅시다!

곡의 특성들 간의 상관관계를 찾아서 어떤 요소들이 곡의 인기에 영향을 미치는지 결정하고 싶습니다.

이를 통해 변수들이 연결되어 있는지, 그리고 그 연결의 강도가 어느 정도인지를 이해할 수 있습니다.

In [ ]:
# 다시 몇 가지 문자열을 제거합니다. 맞습니다. 컴퓨터는 문자열을 좋아하지 않습니다 😁

corr_data = data.drop(["spotify_id", "snapshot_date", "album_name", "name", "artists", "album_release_date"], axis=1)

In [ ]:
# 먼저 '국가'와 다른 변수들의 변화 간의 연관성을 확인해봅시다

corr = corr_data.corr()['country'].sort_values(ascending = False)
corr = corr.to_frame()
corr.style.background_gradient(cmap="RdYlBu")

보신 것처럼 상관관계 값의 범위는 **-1에서 1 사이**입니다:  
- **양수 (1에 가까운 값)**: 강한 직접적 관계를 나타냅니다 (한 변수가 증가하면 다른 변수도 증가합니다).  
- **음수 (-1에 가까운 값)**: 역의 관계를 나타냅니다 (하나가 증가하면 다른 하나는 감소합니다).  
- **0에 가까운 값**: 상관관계가 거의 없음을 의미합니다.  

히트맵은 **Red-Yellow-Blue (RdYlBu) 색상 척도**를 사용합니다:  
- **빨강**: 강한 음의 상관관계를 나타냅니다.  
- **파랑**: 강한 양의 상관관계를 나타냅니다.  
- **노랑**: 약한 상관관계나 상관관계 없음을 강조합니다.  

시각화를 다시 확인하면 한눈에 패턴과 관계를 찾을 수 있습니다! 🔍 

이제 `popularity` (인기도)도 확인해봅시다.

In [ ]:
# 이제 인기도를 확인해봅시다

corr = corr_data.corr()['popularity'].sort_values(ascending = False)
corr = corr.to_frame()
corr.style.background_gradient(cmap="RdYlBu")

In [3]:
## See what's more correlated? What does the data tell you? 

### 더 많은 상관관계가 있나요? 데이터가 무엇을 말해주나요?

In [ ]:
import sys
import os
sys.path.append(os.path.abspath('../.dontlookhere/'))
from quiz1 import *

In [ ]:
quiz_eda()

### 히트맵!

히트맵은 개별 값이 색상으로 표현되는 데이터의 그래픽 표현입니다. 일반적으로 행렬 형식의 값 강도를 시각화하는 데 사용되며, 변수 간의 패턴이나 관계를 더 쉽게 해석할 수 있게 합니다.

데이터 분석에서 히트맵은 데이터 세트의 상관관계 행렬을 시각화하는 데 자주 사용됩니다. 상관관계 행렬은 데이터 세트의 피처(특성)들 간의 쌍을 이루는 상관관계 계수를 보여주는 표입니다. 히트맵은 이러한 상관관계 값을 색칠하여 서로 다른 피처 간의 관계를 직관적으로 이해할 수 있는 방법을 제공합니다.

아래에서 데이터 세트의 히트맵을 만들어 곡의 특성들 간의 관계를 한눈에 파악해 봅시다. 어떤 피처들이 강하게 상관되어 있는지 (양의 상관관계든 음의 상관관계든) 빠르게 볼 수 있습니다.

In [ ]:
plt.figure(figsize = (20, 10))
sns.heatmap(corr_data.corr(), annot = True, cmap='RdYlBu')
plt.show()

### 히트맵은 피처 선택에 도움이 됩니다

상관관계 히트맵으로 곡의 특성들을 시각화하면, 모델에 포함할 곡의 특성(데이터 피처)을 선택할 수 있습니다. 목표 변수(country, 국가)에 강한 상관관계를 가진 피처가 더 유용할 수 있으며, 서로 높은 상관관계를 가진 피처들은 중복될 수 있습니다.

지도를 다시 보고 모델 개발을 위해 사용해야 할 중요한 피처들을 생각해 봅시다!

🌍 각 국가의 피처 평균값을 살펴봅시다 - 국가는 이제 숫자로 표현된다는 것을 기억하세요.

In [ ]:
corr_data.groupby('country').mean().reset_index()

모국가가 더 나은 취향을 가지고 있는 것처럼 보이거나 인기도 목록에 큰 영향을 미치고 있는 것 같습니다 🙈

오른쪽으로 스크롤하면 `popularity` (인기도) 열을 볼 수 있습니다. 더 높은 인기도 값을 가진 국가는 차트에 더 전 지구적으로 인정받는 곡들을 가지는 경향이 있습니다. 따라서 그 국가는 인기 있는 곡들이 전 지구적 트렌드와 더 밀접하게 일치하기 때문에(더 높은 인기도) "더 나은 취향"을 가지고 있을 수 있습니다 :)

이제 특정 기간에 인기 있는 곡들이 무엇인지 살펴봅시다 👇

In [ ]:
data['snapshot_date'] = pd.to_datetime(data['snapshot_date'])

start_date = pd.Timestamp(2023, 12, 20)
end_date = pd.Timestamp(2024, 1, 1)
filtered_data = data[(data['snapshot_date'] >= start_date) & (data['snapshot_date'] < end_date)]

# 곡별로 그룹화하고 각 곡의 평균 인기도를 계산합니다
popularity_per_song = filtered_data.groupby('name')['popularity'].mean()

# 곡을 인기도 내림차순으로 정렬하고 상위 10개를 선택합니다
top_10_songs = popularity_per_song.nlargest(10).reset_index()['name']

print("2023년 12월 20일부터 2024년 1월 1일까지의 상위 10곡:")
print(top_10_songs)

크리스마스 노래들! 🎅 놀랍지 않네요 :)

### 퀴즈 시간 🤓

In [ ]:
quiz_heatmap()

🦄 이제 데이터를 이해하고 '어떤 곡의 특성이 특정 국가에 더 인기 있을까'를 결정하기 위한 핵심 특성들을 파악했으므로, 데이터 사이언스에 뛰어들 수 있습니다! 흥미롭지 않나요?

다음 폴더로 이동해 봅시다.

왼쪽 메뉴에서 `🗂️/jukebox`를 클릭하여 한 폴더 위로 이동한 다음, `2-dev_datascience` 폴더로 이동하여 첫 번째 노트북인 [1-experiment_train.ipynb](../2-dev_datascience/1-experiment_train.ipynb)을 열고 계속 진행하세요 :)